<a href="https://colab.research.google.com/github/Ramprashanth17/Gen_AI/blob/main/rag-learning-platform/notebooks/2_Chunking_and_Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Tutorial 02: Document Chunking and Tokenization

**Learning Objectives:**
- Understand why RAG systems need document chunking
- Master the difference between chunking, tokenization, and embedding
- Learn different chunking strategies and when to use each
- Implement production-ready chunking with optimal parameters

**Prerequisites:**
- ✅ Completed `01_embeddings_fundamentals.ipynb`
- ✅ Understand embeddings and cosine similarity

**Time Required:** 30-40 minutes

---

## 📖 Table of Contents
1. [The Problem: Why Chunk?](#problem)
2. [The Three-Step Pipeline](#pipeline)
3. [Chunking Strategies](#strategies)
4. [Tokenization Deep Dive](#tokenization)
5. [Best Practices](#practices)
6. [Practice Exercises](#exercises)

---
**Author:** Ramprashanth | INFO 7390 Final Project | December 2025

In [1]:
# Install packages
!pip install -q sentence-transformers tiktoken scikit-learn google-generativeai

In [2]:
# ═══════════════════════════════════════════════════════
# 📁 SETUP
# ═══════════════════════════════════════════════════════
# Imports
import google.generativeai as genai
from google.colab import userdata  # <--- THIS IS MISSING
from sentence_transformers import SentenceTransformer
import tiktoken
import re

# 1. Fetch the key from Colab Secrets
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
    print(f"⚠️ Error fetching key: {e}")
    print("Please ensure you created a secret named 'GOOGLE_API_KEY' in the 🔑 tab on the left.")

# 2. Test the connection
try:
    model = genai.GenerativeModel('gemini-2.5-flash')
    response = model.generate_content("Say 'Setup successful!' if you can read this.")
    print(response.text)
    print("✅ Setup complete! Let's explore chunking and tokenization.")
except Exception as e:
    print(f"❌ Setup failed: {e}")

Setup successful!
✅ Setup complete! Let's explore chunking and tokenization.


## 🧩 Process-2: The Art of Chunking (Breaking it Down)

**The Problem: The "Goldilocks" Dilemma**
We now know how to turn text into numbers (vectors). But here is the practical challenge:
* If we embed an **entire document** (like a 50-page PDF) as one single vector, the specific meanings get "diluted." The vector tries to represent everything, so it represents nothing well.
* If we embed **single words**, we lose all context (e.g., "bank" could mean a river bank or a finance bank).

For Example:
Imagine you have a **100-page employee handbook**. A user asks:

> "What's the company's remote work policy?"

### ❌ Approach 1: Send Entire Handbook to LLM
```
100 pages → ~75,000 tokens → Send to LLM
```

**Problems:**

- 💰 **Expensive:** \$0.075 per query (at $0.001/1k tokens)
- ⏰ **Slow:** Takes 30+ seconds to process
- 🔊 **Noisy:** 99 pages are irrelevant!
- ⚠️ **Hit limits:** Exceeds many model context windows
- 😕 **Poor answers:** LLM gets confused by too much info

---

### ✅ Approach 2: Send Only Relevant Paragraph
```
1 paragraph → ~100 tokens → Send to LLM
```

**Benefits:**
- 💰 **Cheap:** $0.0001 per query (750x cheaper!)
- ⚡ **Fast:** Processes in 2 seconds
- 🎯 **Precise:** Only relevant information
- ✅ **Better answers:** LLM focuses on what matters

---

We need something "just right." We need **Chunks**.

**What is Chunking?**

**Chunking** is the process of breaking large documents into smaller, meaningful pieces that can be:
1. **Embedded** individually (each chunk gets its own vector)
2. **Searched** semantically (find relevant chunks)
3. **Retrieved** efficiently (send only what's needed)

**Why is this crucial for RAG?**
1.  **Precision:** When a user asks "What is the return policy?", you want to retrieve the specific *paragraph* about returns, not the whole company handbook.
2.  **Context Limits:** LLMs (The Generator) have a limit on how much text they can read. Providing small, relevant chunks is much more efficient than feeding in massive documents.

**The Analogy:**
Imagine you are studying for an exam. You don't try to memorize the entire textbook as one giant blob of text.
Instead, you make **Flashcards**.
* **The Document:** The whole Textbook.
* **The Chunks:** The Flashcards.

In the next step, we will take our raw text and slice it into these "flashcards" so our embedding model can index them effectively.

In [4]:
# Install sentence transformers

!pip install -q sentence-transformers scikit-learn

<a id="pipeline"></a>
## 🔄 The Three-Step Pipeline: Chunking → Tokenization → Embedding

**CRITICAL:** These are THREE DIFFERENT steps! Don't confuse them!
```
📄 LARGE DOCUMENT (10,000 words)
        ↓
    ╔══════════════════════════════════════╗
    ║   STEP 1: CHUNKING (You control!)   ║
    ╚══════════════════════════════════════╝
        ↓
    Chunk 1 (500 words) | Chunk 2 (500 words) | Chunk 3 (500 words)...
        ↓
    ╔══════════════════════════════════════╗
    ║ STEP 2: TOKENIZATION (Model does!)  ║
    ╚══════════════════════════════════════╝
        ↓
    Token IDs: [40, 3021, 12875, ...] for each chunk
        ↓
    ╔══════════════════════════════════════╗
    ║   STEP 3: EMBEDDING (Model does!)   ║
    ╚══════════════════════════════════════╝
        ↓
    Vectors: [0.234, 0.891, 0.123, ...] (384 numbers) for each chunk
        ↓
    ✅ Ready for storage and retrieval!
```

---

### Understanding Each Step:

| Step | What | Who Does It | Purpose |
|------|------|-------------|---------|
| **1. Chunking** | Split document into pieces | **YOU** (developer) | Make manageable pieces |
| **2. Tokenization** | Break text into tokens | **MODEL** (automatic) | Prepare for processing |
| **3. Embedding** | Convert to vectors | **MODEL** (automatic) | Capture semantic meaning |

**Key Point:** Chunking happens FIRST, before anything else!

In [5]:
# ═══════════════════════════════════════════════════════
# 🔬 DEMO: The Complete Pipeline
# ═══════════════════════════════════════════════════════

document = """Dogs are loyal companions that require proper care.
They need daily exercise and a balanced diet.
Training from an early age is essential for good behavior."""

print("="*70)
print("📄 ORIGINAL DOCUMENT")
print("="*70)
print(document)
print(f"\n📏 Length: {len(document)} characters\n")

# ══════════════════════════════════════════════════════════
# STEP 1: CHUNKING (You do this - Split into pieces)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STEP 1: CHUNKING (Breaking document into pieces)")
print("="*70)

# Simple chunking by sentences
chunks = [s.strip() + '.' for s in document.split('.') if s.strip()]

print(f"✅ Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: '{chunk}'")
    print(f"  → {len(chunk)} characters\n")

# ══════════════════════════════════════════════════════════
# STEP 2: TOKENIZATION (Model does this - Break into tokens)
# ══════════════════════════════════════════════════════════

print("="*70)
print("STEP 2: TOKENIZATION (Model breaks text into tokens)")
print("="*70)

tokenizer = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer

print("\nTokenizing Chunk 1...\n")
chunk_1 = chunks[0]
tokens = tokenizer.encode(chunk_1)

print(f"📝 Text: '{chunk_1}'")
print(f"🔢 Token count: {len(tokens)}")
print(f"🆔 Token IDs: {tokens}\n")

print("Token-by-token breakdown:")
print("-"*70)
for i, token_id in enumerate(tokens):
    token_text = tokenizer.decode([token_id])
    print(f"  Token {i:2d}: ID {token_id:5d} → '{token_text}'")

# ══════════════════════════════════════════════════════════
# STEP 3: EMBEDDING (Model converts to semantic vectors)
# ══════════════════════════════════════════════════════════

print("\n" + "="*70)
print("STEP 3: EMBEDDING (Converting to semantic vectors)")
print("="*70)

model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nEmbedding all chunks...\n")
print("-"*70)

for i, chunk in enumerate(chunks, 1):
    embedding = model.encode(chunk)
    token_count = len(tokenizer.encode(chunk))

    print(f"Chunk {i}:")
    print(f"  📝 Text length: {len(chunk)} characters")
    print(f"  🔢 Token count: {token_count} tokens")
    print(f"  🎯 Embedding size: {len(embedding)} numbers (ALWAYS 384!)")
    print(f"  📊 First 5 values: {embedding[:5]}")
    print()

print("="*70)
print("🎯 KEY INSIGHT:")
print("Input size varies → Token count varies → Embedding ALWAYS 384!")
print("="*70)

📄 ORIGINAL DOCUMENT
Dogs are loyal companions that require proper care. 
They need daily exercise and a balanced diet. 
Training from an early age is essential for good behavior.

📏 Length: 158 characters

STEP 1: CHUNKING (Breaking document into pieces)
✅ Created 3 chunks

Chunk 1: 'Dogs are loyal companions that require proper care.'
  → 51 characters

Chunk 2: 'They need daily exercise and a balanced diet.'
  → 45 characters

Chunk 3: 'Training from an early age is essential for good behavior.'
  → 58 characters

STEP 2: TOKENIZATION (Model breaks text into tokens)

Tokenizing Chunk 1...

📝 Text: 'Dogs are loyal companions that require proper care.'
🔢 Token count: 10
🆔 Token IDs: [35, 27403, 527, 29947, 41957, 430, 1397, 6300, 2512, 13]

Token-by-token breakdown:
----------------------------------------------------------------------
  Token  0: ID    35 → 'D'
  Token  1: ID 27403 → 'ogs'
  Token  2: ID   527 → ' are'
  Token  3: ID 29947 → ' loyal'
  Token  4: ID 41957 → ' companion

In [6]:
# EXPERIMENT: Different Chunking Strategies

# Sample document
document = """
Chapter 1: Dog Care Basics

Dogs are loyal companions that require proper care and attention.
They need daily exercise to stay healthy and happy. A typical dog
should get at least 30 minutes of physical activity each day.

Nutrition is another crucial aspect of dog care. Dogs require a
balanced diet with proper amounts of protein, fats, and carbohydrates.
The exact nutritional needs vary by breed, age, and activity level.

Training is essential for a well-behaved dog. Start training early
with basic commands like sit, stay, and come. Positive reinforcement
works best for most dogs.

Chapter 2: Common Health Issues

Dogs can face various health challenges throughout their lives.
Regular vet checkups are important for early detection of problems.
Common issues include dental disease, obesity, and joint problems.
"""

print("🔬 CHUNKING EXPERIMENT")
print("="*60)

# Strategy 1: Fixed Character Count (NO OVERLAP)
def chunk_by_chars_no_overlap(text, chunk_size=200):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

# Strategy 2: Fixed Character Count (WITH OVERLAP)
def chunk_by_chars_with_overlap(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)  # Move forward, but keep overlap
    return chunks

# Strategy 3: Sentence-Based (Semantic)
def chunk_by_sentences(text, sentences_per_chunk=3):
    import re
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = '. '.join(sentences[i:i+sentences_per_chunk]) + '.'
        chunks.append(chunk)
    return chunks

# Strategy 4: Paragraph-Based (Natural Boundaries)
def chunk_by_paragraphs(text):
    paragraphs = text.split('\n\n')
    return [p.strip() for p in paragraphs if p.strip()]

# Test all strategies
print("\n📊 STRATEGY 1: Fixed chars, NO overlap")
print("-"*60)
chunks1 = chunk_by_chars_no_overlap(document, chunk_size=150)
print(f"Number of chunks: {len(chunks1)}")
print(f"Chunk 3 (middle): '{chunks1[2][:80]}...'")
print("❌ Problem: Notice how it cuts mid-sentence!")

print("\n📊 STRATEGY 2: Fixed chars, WITH overlap (50 chars)")
print("-"*60)
chunks2 = chunk_by_chars_with_overlap(document, chunk_size=150, overlap=50)
print(f"Number of chunks: {len(chunks2)}")
print(f"Overlap example:")
print(f"  Chunk 2 end: '...{chunks2[1][-50:]}'")
print(f"  Chunk 3 start: '{chunks2[2][:50]}...'")
print("✅ Better: Context preserved at boundaries!")

print("\n📊 STRATEGY 3: By sentences (3 per chunk)")
print("-"*60)
chunks3 = chunk_by_sentences(document, sentences_per_chunk=2)
print(f"Number of chunks: {len(chunks3)}")
print(f"Chunk 1: '{chunks3[0]}'")
print("✅ Good: Complete thoughts preserved!")

print("\n📊 STRATEGY 4: By paragraphs (semantic boundaries)")
print("-"*60)
chunks4 = chunk_by_paragraphs(document)
print(f"Number of chunks: {len(chunks4)}")
for i, chunk in enumerate(chunks4[:3], 1):
    print(f"Chunk {i} ({len(chunk)} chars): '{chunk[:60]}...'")
print("✅ Best: Natural semantic units!")

print("\n" + "="*60)
print("🎯 KEY TAKEAWAY:")
print("Semantic chunking (paragraphs/sections) > Character chunking")
print("But character chunking with overlap is still useful!")

🔬 CHUNKING EXPERIMENT

📊 STRATEGY 1: Fixed chars, NO overlap
------------------------------------------------------------
Number of chunks: 6
Chunk 3 (middle): 't with proper amounts of protein, fats, and carbohydrates.
The exact nutritional...'
❌ Problem: Notice how it cuts mid-sentence!

📊 STRATEGY 2: Fixed chars, WITH overlap (50 chars)
------------------------------------------------------------
Number of chunks: 9
Overlap example:
  Chunk 2 end: '...cal activity each day.

Nutrition is another cruci'
  Chunk 3 start: 'cal activity each day.

Nutrition is another cruci...'
✅ Better: Context preserved at boundaries!

📊 STRATEGY 3: By sentences (3 per chunk)
------------------------------------------------------------
Number of chunks: 6
Chunk 1: 'Chapter 1: Dog Care Basics

Dogs are loyal companions that require proper care and attention. They need daily exercise to stay healthy and happy.'
✅ Good: Complete thoughts preserved!

📊 STRATEGY 4: By paragraphs (semantic boundaries)
-----

<a id="tokenization"></a>
## 🔤 Deep Dive: What is Tokenization?

### Definition

**Tokenization** is the process of breaking text into small units (tokens) that a model can process.

Think of it like breaking sentences into words, but smarter!

---

### How Tokenizers Work
```python
Text: "I love dogs!"

Tokenizer breaks it into:
["I", " love", " dogs", "!"]

Each gets an ID:
[40, 3021, 12875, 0]
```

**Why IDs?** Models only understand numbers, not text!

---

### Different Tokenizers

| Tokenizer | Used By | Tokens for "I love dogs!" |
|-----------|---------|---------------------------|
| **cl100k_base** | GPT-4, GPT-3.5 | 4 tokens |
| **p50k_base** | GPT-3 | 4 tokens |
| **SentenceTransformer** | MiniLM, MPNet | Internal |

---

### Why Token Count Matters

**Billing:** Most AI APIs charge per token!
### Why Token Count Matters

**Billing:** Most AI APIs charge per token!

**Example 1: Without RAG (Direct Query)**
```python
Query: "I love dogs!" (4 tokens)
Response: "Dogs are wonderful!" (4 tokens)
Total: 8 tokens × $0.001/1k = $0.000008
Cost per query: $0.000008 ✅ Cheap!
```

**Example 2: With RAG (Context-Enhanced)**
```python
Query: "What do dogs need?" (5 tokens)
Retrieved Context:
  - Chunk 1: Exercise info (120 tokens)
  - Chunk 2: Nutrition info (95 tokens)
  - Chunk 3: Training info (110 tokens)
Total Context: 325 tokens

Input to LLM: Query (5) + Context (325) = 330 tokens
Response: "Dogs need exercise, nutrition..." (15 tokens)
Total: 345 tokens × $0.001/1k = $0.000345

Cost per query: $0.000345 (43x more expensive!)
```

**Trade-off:**
- Pay more → Get accurate, sourced answers ✅
- Pay less → Get generic, possibly wrong answers ❌

**This is why chunk size optimization matters!**
- Too big chunks = wasted tokens = higher cost
- Too small chunks = missing context = poor answers
- **Sweet spot: 512 tokens per chunk, retrieve 3-5 chunks**
```

**This is why chunk size matters!**

In [7]:
# ═══════════════════════════════════════════════════════
# 🔬 EXPERIMENT: Tokenization Patterns
# ═══════════════════════════════════════════════════════

tokenizer = tiktoken.get_encoding("cl100k_base")

test_texts = [
    "dog",
    "dogs",
    "puppy",
    "I love dogs",
    "I absolutely adore dogs!",
    "Dogs are loyal companions",
    "🐕"  # Emoji!
]

print("🔬 TOKENIZATION EXPERIMENTS")
print("="*70)

for text in test_texts:
    tokens = tokenizer.encode(text)
    token_details = [tokenizer.decode([t]) for t in tokens]

    print(f"\n📝 Text: '{text}'")
    print(f"🔢 Token count: {len(tokens)}")
    print(f"🆔 Token IDs: {tokens}")
    print(f"📋 Tokens: {token_details}")

print("\n" + "="*70)
print("🎯 OBSERVATIONS:")
print("- Spaces are part of tokens (' dogs' vs 'dogs')")
print("- Similar words have different tokens ('dog' vs 'dogs')")
print("- Longer sentences = more tokens")
print("- Emojis can be multiple tokens!")
print("="*70)

🔬 TOKENIZATION EXPERIMENTS

📝 Text: 'dog'
🔢 Token count: 1
🆔 Token IDs: [18964]
📋 Tokens: ['dog']

📝 Text: 'dogs'
🔢 Token count: 1
🆔 Token IDs: [81134]
📋 Tokens: ['dogs']

📝 Text: 'puppy'
🔢 Token count: 2
🆔 Token IDs: [79, 65129]
📋 Tokens: ['p', 'uppy']

📝 Text: 'I love dogs'
🔢 Token count: 3
🆔 Token IDs: [40, 3021, 12875]
📋 Tokens: ['I', ' love', ' dogs']

📝 Text: 'I absolutely adore dogs!'
🔢 Token count: 5
🆔 Token IDs: [40, 11112, 61735, 12875, 0]
📋 Tokens: ['I', ' absolutely', ' adore', ' dogs', '!']

📝 Text: 'Dogs are loyal companions'
🔢 Token count: 5
🆔 Token IDs: [35, 27403, 527, 29947, 41957]
📋 Tokens: ['D', 'ogs', ' are', ' loyal', ' companions']

📝 Text: '🐕'
🔢 Token count: 3
🆔 Token IDs: [9468, 238, 243]
📋 Tokens: ['�', '�', '�']

🎯 OBSERVATIONS:
- Spaces are part of tokens (' dogs' vs 'dogs')
- Similar words have different tokens ('dog' vs 'dogs')
- Longer sentences = more tokens
- Emojis can be multiple tokens!


<a id="practices"></a>
## 📏 Best Practices for Production RAG

### Industry-Standard Parameters
```python
# Recommended settings
CHUNK_SIZE = 512        # tokens (~400 words)
OVERLAP = 128           # tokens (25% overlap)
MAX_CHUNKS_RETRIEVED = 3 # per query
```

---

### Why These Numbers?

#### **Chunk Size: 512 tokens**

✅ **Fits model limits:** Most embedding models support 512 tokens  
✅ **Complete thoughts:** Usually captures full paragraphs  
✅ **Not too small:** Preserves context  
✅ **Not too large:** Maintains precision  

#### **Overlap: 128 tokens (25%)**

✅ **Preserves context:** Information at boundaries isn't lost  
✅ **Cost-effective:** 25% overhead is acceptable  
✅ **Tested standard:** Proven across many RAG systems  

---

### Cost Analysis

**Example: 10,000-word document**

| Strategy | Tokens Stored | Embedding Cost | Quality |
|----------|---------------|----------------|---------|
| **No chunks** | Truncated at 512 | \$0.0001 | ⭐ Poor |
| **No overlap** | ~7,500 | \$0.00075 | ⭐⭐⭐ OK |
| **25% overlap** | ~9,375 | \$0.00094 | ⭐⭐⭐⭐⭐ Best |

**Verdict:** 25% more cost for 50% better quality = Worth it!

---

### Chunking Decision Tree
```
Is content structured (sections/paragraphs)?
├─ YES → Chunk by sections/paragraphs ✅ BEST
└─ NO → Is it conversational (chat/Q&A)?
    ├─ YES → Chunk by conversation turns
    └─ NO → Use fixed size with 25% overlap
```

## ⚠️ Common Doubts & Pitfalls

### ❓ Doubt 1: "I retrieve only 3-5 chunks. What if there are 10 relevant chunks?"

**Short Answer:** You're trading completeness for cost and focus.

**What Happens:**
```python
# Your document has 100 chunks total
# Query: "What do dogs need?"
# 10 chunks are actually relevant!

# But you set top_k=3
results = vector_db.search(query, top_k=3)
# Returns: Top 3 MOST similar chunks only

# The other 7 relevant chunks? Ignored! ⚠️
```

**Why This is OK (Usually):**
- ✅ **Best info first:** Vector search ranks by similarity - top 3 usually have the answer
- ✅ **Cost control:** 3 chunks = ~1,500 tokens; 10 chunks = ~5,000 tokens (3.3x more!)
- ✅ **LLM focus:** Too much context confuses LLMs ("needle in haystack" problem)

**When to Increase top_k:**
```python
# Complex queries needing multiple perspectives
top_k = 5-7  # Research, comparison, comprehensive analysis

# Simple factual queries
top_k = 2-3  # "What's the return policy?" - one chunk usually enough

# Budget unlimited, need everything
top_k = 10+  # Enterprise use cases
```

**Pro Tip:** If answers seem incomplete, try `top_k=5` first before going higher!

---

### ❓ Doubt 2: "Why does 'Dogs' tokenize differently than 'dogs'?"

**Short Answer:** Capitalization affects token frequency, so tokenizers split rare patterns.

**The Pattern:**
```python
"dogs" → [12875]           # Common word → Single token ✅
"Dogs" → [35, 12875]       # Less common → "D" + "ogs" ⚠️
"DOGS" → [35, 12501]       # Rare (shouting) → Splits further ⚠️
```

**Why This Matters:**
```python
text = "DOGS NEED EXERCISE!"  # All caps (10 tokens)
text = "dogs need exercise"    # Lowercase (3 tokens)

# Same meaning, 3x token difference = 3x cost! 💰
```

**Solution:** Normalize text before chunking (usually lowercase, except proper nouns).

---

### ❓ Doubt 3: "Is RAG context size fixed at 500 tokens?"

**Short Answer:** NO! You control the total context size.

**You Control:**
```python
chunk_size = 512 tokens      # Size of each chunk
top_k = 3                    # How many chunks to retrieve

Total context = 512 × 3 = 1,536 tokens

# Want less? Reduce either value:
chunk_size = 256, top_k = 3  → 768 tokens (cheaper!)
chunk_size = 512, top_k = 2  → 1,024 tokens

# Want more? Increase:
chunk_size = 512, top_k = 5  → 2,560 tokens (better quality, higher cost)
```

**Typical Budgets:**
```
💰 Budget Mode:     ~800 tokens   (2 chunks × 400 tokens)
⚖️ Balanced:        ~1,500 tokens (3 chunks × 512 tokens) ✅ RECOMMENDED
💎 High Quality:    ~3,000 tokens (6 chunks × 512 tokens)
🏢 Enterprise:      ~8,000 tokens (15+ chunks)
```

---

### ❓ Doubt 4: "Should I chunk before or after cleaning text?"

**Short Answer:** Clean FIRST, then chunk!

**Wrong Order:**
```python
❌ BAD FLOW:
1. Chunk raw text (preserves weird formatting)
2. Clean each chunk (inconsistent results)
3. Embed (embeddings vary due to formatting noise)
```

**Right Order:**
```python
✅ GOOD FLOW:
1. Clean text (remove extra spaces, normalize, etc.)
2. Chunk cleaned text (consistent boundaries)
3. Embed (clean embeddings)

# Example
text = load_document()
text = text.strip()                    # Remove leading/trailing spaces
text = re.sub(r'\s+', ' ', text)      # Normalize whitespace
text = text.replace('\r\n', '\n')     # Normalize line breaks
chunks = chunk_by_paragraphs(text)    # NOW chunk!
```

---

### ❓ Doubt 5: "What if my document is smaller than chunk_size?"

**Short Answer:** That's fine! Use the whole document as one chunk.
```python
def smart_chunk(text, chunk_size=512):
    """Handle documents of any size"""
    tokenizer = tiktoken.get_encoding("cl100k_base")
    token_count = len(tokenizer.encode(text))
    
    if token_count <= chunk_size:
        # Document is small enough - use as is!
        return [text]
    else:
        # Document is large - chunk it
        return chunk_by_paragraphs(text)

# Examples
small_doc = "Dogs need exercise."  # 4 tokens
result = smart_chunk(small_doc, chunk_size=512)
# Returns: ["Dogs need exercise."]  # One chunk ✅

large_doc = "..." * 1000  # 3000 tokens
result = smart_chunk(large_doc, chunk_size=512)
# Returns: [chunk1, chunk2, chunk3, chunk4, chunk5, chunk6]  # Multiple chunks
```

**No minimum size required!**

---

### ❓ Doubt 6: "Isn't overlap just wasting money?"

**Short Answer:** No! It's an investment in quality.

**Cost-Benefit Analysis:**
```python
# 1000-word document, 512-token chunks

WITHOUT OVERLAP:
Chunks: 2
Tokens to embed: ~750
Cost: $0.00075
Quality: 7/10 (context breaks at boundaries) ⚠️

WITH 25% OVERLAP:
Chunks: 3
Tokens to embed: ~940
Cost: $0.00094 (25% more)
Quality: 9/10 (context preserved) ✅

# Verdict: Pay 25% more for 30% better quality!
```

**When Overlap REALLY Helps:**
```python
# Without overlap:
Chunk 1: "...Dogs need daily exercise. They"
Chunk 2: "should get 30 minutes of activity..."

Query: "How much exercise do dogs need?"
Retrieved: Chunk 1 only → "They" what? Incomplete! ❌

# With 25% overlap:
Chunk 1: "...Dogs need daily exercise. They should get 30 minutes..."
Chunk 2: "They should get 30 minutes of activity per day..."

Query: "How much exercise do dogs need?"
Retrieved: Either chunk → Complete answer! ✅
```

**Skip overlap only if:** Documents are naturally separated (Q&A pairs, bullet points, distinct sections).

---

### ❓ Doubt 7: "How do I know if my chunks are 'good'?"

**Short Answer:** Test with real queries!

**Quick Quality Checklist:**
```python
✅ GOOD CHUNK:
- Contains complete thought/idea
- Makes sense if read alone
- Has clear context (not "It needs..." but "Dogs need...")
- 200-600 tokens (sweet spot)

❌ BAD CHUNK:
- Cuts mid-sentence: "Dogs need exercise and"
- No context: "They need 30 minutes" (who is "they"?)
- Too small: "Dogs." (no useful info)
- Too large: Entire 5-page chapter (too broad)
```

**Testing Method:**
```python
# 1. Create test queries
queries = [
    "What do dogs need?",
    "How to train puppies?",
    "Dog health issues?"
]

# 2. Retrieve chunks
for query in queries:
    results = retrieve_chunks(query, top_k=3)
    
    # 3. Check manually:
    print(f"\nQuery: {query}")
    for i, chunk in enumerate(results, 1):
        print(f"Chunk {i}: {chunk[:100]}...")
        # Ask yourself: Does this answer the query?

# 4. If chunks seem incomplete or irrelevant:
#    → Adjust chunk_size or chunking strategy
```

**Rule of Thumb:** If you can't answer the query from the retrieved chunks, your chunking needs adjustment!

---

### 🎯 Key Takeaways

1. **top_k=3-5 is usually enough** - more isn't always better (cost vs quality)
2. **Tokenization varies** - lowercase is more efficient than capitals
3. **RAG context is NOT fixed** - you control chunk_size × top_k
4. **Clean BEFORE chunking** - consistent results
5. **Small documents are OK** - use as single chunk
6. **Overlap isn't waste** - it's quality insurance (25% cost for 30% better)
7. **Test your chunks** - if queries fail, adjust strategy

**Remember:** Good chunking is about finding the balance between:
- 💰 Cost (fewer/smaller chunks)
- 🎯 Quality (more/overlapping chunks)
- ⚡ Speed (less context to process)

**Start with:** 512 tokens, 25% overlap, top_k=3, then optimize based on your use case!



<a id="exercises"></a>
## 🧪 Practice Exercises

### Exercise 1: Token Counter

**Task:** Create a function that counts tokens and estimates cost.

<details>
<summary>💡 Hint</summary>

Use tiktoken to encode the text, then calculate:
- Token count = len(encoded)
- Cost = (token_count / 1000) * cost_per_1k
</details>

In [ ]:
def estimate_cost(text, cost_per_1k=0.001):
    """
    Calculate tokens and cost for embedding text.

    Args:
        text: String to analyze
        cost_per_1k: Cost per 1000 tokens

    Returns:
        dict with token_count and cost
    """
    # YOUR CODE HERE
    pass

# Test it
text = "Your test text here..."
result = estimate_cost(text)
print(f"Tokens: {result['tokens']}, Cost: ${result['cost']:.6f}")

### Exercise 2: Smart Chunker

**Task:** Build an adaptive chunking function!

**Requirements:**
1. Try paragraph-based chunking first
2. If paragraphs > 600 characters, split by sentences
3. Add 20% overlap
4. Return list of chunks with metadata

In [ ]:
def smart_chunk(text, max_size=600, overlap_pct=0.20):
    """
    Intelligently chunk text with adaptive strategy.

    Returns:
        List of dicts: [{'text': chunk, 'tokens': count, 'strategy': method}, ...]
    """
    # YOUR CODE HERE
    pass

# Test
doc = """[Long test document]"""
chunks = smart_chunk(doc)
for i, chunk_info in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk_info['tokens']} tokens ({chunk_info['strategy']})")

### Exercise 3: Chunk Visualizer

**Task:** Visualize chunk boundaries and overlaps!

Create a visual representation showing:
- Where each chunk starts/ends
- Overlapping regions
- Token counts

**Bonus:** Use colors to show overlap regions!

### Solutions

<details>
<summary>Click to reveal solutions</summary>

[Solutions will be provided separately - try yourself first!]

</details>

## ✅ Summary: What You Learned

### 🎯 Key Concepts

**1. The Three Steps (Don't confuse them!)**
```
Chunking → Tokenization → Embedding
 (YOU)      (MODEL)        (MODEL)
```

**2. Why Chunking Matters**
- ✅ Makes documents searchable at paragraph-level
- ✅ Reduces cost (send only relevant info)
- ✅ Improves answer quality (less noise)
- ✅ Handles model limits (512 token max)

**3. Chunking Strategies**
- 🥇 **Best:** Semantic (paragraphs/sections)
- 🥈 **Good:** Sentences with overlap
- 🥉 **Fallback:** Fixed size with overlap

**4. Production Standards**
- **Chunk size:** 512 tokens
- **Overlap:** 25% (128 tokens)
- **Cost increase:** ~25% for much better quality

---

### 📊 Pipeline Recap
```python
# This is what happens in a RAG system:

# 1. You chunk the document
chunks = chunk_document(text, size=512, overlap=128)

# 2. Model tokenizes each chunk (internal)
# tokens = tokenizer.encode(chunk)  # Happens inside model

# 3. Model embeds each chunk (internal)
embeddings = model.encode(chunks)  # Returns 384D vectors

# 4. You store in vector database
db.add(chunks, embeddings)

# 5. User queries → Retrieve relevant chunks
results = db.search(query, top_k=3)

# 6. Send to LLM for final answer
answer = llm.generate(query, context=results)
```

---



### 🚀 What's Next?

Now that you understand chunking and tokenization, we'll learn how to **store** these embeddings in a vector database!

**Next Tutorial:** `03_vector_storage_chromadb.ipynb`

Topics covered:
- What are vector databases?
- Setting up ChromaDB
- Storing and querying embeddings
- Building the retrieval system

---

### 📚 Additional Resources

- [Tiktoken Documentation](https://github.com/openai/tiktoken)
- [Sentence Transformers Guide](https://www.sbert.net/)
- [RAG Best Practices Paper](https://arxiv.org/abs/2005.11401)

---

**Questions or feedback?**
- GitHub: https://github.com/Ramprashanth17/Gen_AI/tree/main/rag-learning-platform
- Email: gajarghat.r@northeastern.edu